# Tiny instruction-interface demo (image + task -> manipulable manual)

Give **an image + a task**, get a **step-by-step manual rendered on your real photo**, inside a
digital interface you can **manipulate** - including pressing two things together to join them.

What it demonstrates:
- **Decompose**: task + image -> short ordered atomic steps.
- **Ground (faithful)**: each step drawn as a box/dot on the *real* photo (no re-render, no drift).
- **Flip**: next/prev through steps.
- **Tap-to-deepen**: click an object -> it crops and explains that region.
- **Connect (press to join)**: press the source object, then press the destination -> the interface
  moves those pixels together, marks them "joined", and erases the source's old spot.
- **Multi-image context**: upload a close-up (e.g. a label) -> the model reads it and adds a note.
- **Correct**: switch to "Set highlight" and click to move a highlight by hand.

Model: **Moondream2** (~1.8B). Runs on a free Kaggle **T4 GPU**.
**Before running: Settings -> Accelerator -> GPU T4, and Internet -> On.**

In [ ]:
# 1. Install deps (Kaggle already has torch/torchvision).
#    Do it in ONE resolver pass so numpy isn't left half-upgraded. The previous
#    chain of --force-reinstall calls mixed numpy versions, which caused:
#      ImportError: cannot import name '_center' from 'numpy._core.umath'
#    (numpy 2.x Python source + an out-of-sync compiled umath).
!pip install -q -U "transformers<5" accelerate einops gradio "Pillow==11.2.1" diffusers
# Force ONE clean, self-consistent numpy build (source + compiled umath from the same wheel).
!pip install -q --force-reinstall --no-cache-dir "numpy==2.1.3"

In [ ]:
# 1b. RESTART THE KERNEL so the freshly installed numpy/Pillow load cleanly.
#     This kills the running process; that is expected. After it restarts,
#     run the cells BELOW (you can re-run cell 1 — pip will see everything is
#     already satisfied and won't re-break numpy).
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)
print("Restarting kernel... re-run the cells below after it comes back.")

In [4]:
# 2. Load the small model (Moondream2)
import torch, re
from transformers import AutoModelForCausalLM
from PIL import Image, ImageDraw, ImageFilter
import gradio as gr

# If load ever breaks on the latest code, pin a revision, e.g. revision="2025-04-14"
model = AutoModelForCausalLM.from_pretrained(
    "vikhyatk/moondream2",
    trust_remote_code=True,
    device_map={"": "cuda"},
)
print("model loaded on", next(model.parameters()).device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/277 [00:00<?, ?B/s]

hf_moondream.py: 0.00B [00:00, ?B/s]

vision.py: 0.00B [00:00, ?B/s]

layers.py: 0.00B [00:00, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/vikhyatk/moondream2:
- region.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/vikhyatk/moondream2:
- moondream.py
- lora.py
- region.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/vikhyatk/moondream2:
- hf_moondream.py
- vision.py
- text.py
- utils.py
- moondream.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/3.85G [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/69.0 [00:00<?, ?B/s]

model loaded on cuda:0


In [ ]:
# 2b. Load an image EDITING model so "Connect" GENERATES a joined image (not copy-paste).
#     InstructPix2Pix is small (SD1.5-based) and co-fits with Moondream on a free T4.
!pip install -q diffusers
from diffusers import StableDiffusionInstructPix2PixPipeline

pipe_edit = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    "timbrooks/instruct-pix2pix",
    torch_dtype=torch.float16,
    safety_checker=None,
).to("cuda")
pipe_edit.set_progress_bar_config(disable=True)
print("editor loaded")

In [5]:
# 3. Core logic: decompose, ground, draw, and connect

def generate_steps(image, task, max_steps=5):
    prompt = ('I want to: "' + task + '". Looking at this image, list the physical steps to do it '
              'as a short numbered list. One short single-action sentence per step. Max 5 steps.')
    ans = model.query(image, prompt)["answer"]
    steps = []
    for line in ans.split("\n"):
        line = re.sub(r"^[\-\*\d\.\)\s]+", "", line.strip()).strip()
        if line:
            steps.append(line)
    return steps[:max_steps] or [ans.strip()]

def step_object(image, step):
    q = ('For this instruction step, name the single main object to interact with, '
         'in 1 to 3 words only: "' + step + '"')
    return model.query(image, q)["answer"].strip().strip(".")

def ground(image, obj):
    try:
        objs = model.detect(image, obj).get("objects", [])
        if objs:
            return "box", objs
    except Exception:
        pass
    try:
        pts = model.point(image, obj).get("points", [])
        if pts:
            return "point", pts
    except Exception:
        pass
    return "none", []

def draw_overlay(base, kind, data, label):
    img = base.convert("RGB").copy()
    d = ImageDraw.Draw(img)
    W, H = img.size
    w = max(3, W // 200)
    if kind == "box":
        for b in data:
            d.rectangle([b["x_min"]*W, b["y_min"]*H, b["x_max"]*W, b["y_max"]*H],
                        outline=(255, 60, 60), width=w)
    elif kind == "point":
        for p in data:
            x, y = p["x"]*W, p["y"]*H
            r = max(8, W // 40)
            d.ellipse([x-r, y-r, x+r, y+r], outline=(255, 60, 60), width=w)
    d.rectangle([0, 0, W, 26], fill=(0, 0, 0))
    d.text((6, 7), label[:90], fill=(255, 255, 255))
    return img

def render(canvas, steps, idx):
    if not steps:
        return canvas, "No steps yet."
    step = steps[idx]
    obj = step_object(canvas, step)
    kind, data = ground(canvas, obj)
    view = draw_overlay(canvas, kind, data, "Step " + str(idx+1) + "/" + str(len(steps)) + ": " + obj)
    return view, "Step " + str(idx+1) + "/" + str(len(steps)) + ":  " + step

def erase_region(img, cx, cy, r):
    W, H = img.size
    box = (max(0, cx-r), max(0, cy-r), min(W, cx+r), min(H, cy+r))
    blurred = img.filter(ImageFilter.GaussianBlur(14)).crop(box)
    img.paste(blurred, box)

def connect(canvas, sx, sy, dx, dy):
    # move the source patch onto the destination, erase the old spot, mark as joined
    img = canvas.convert("RGB").copy()
    W, H = img.size
    r = min(W, H) // 7
    src_box = (max(0, sx-r), max(0, sy-r), min(W, sx+r), min(H, sy+r))
    patch = img.crop(src_box)
    erase_region(img, sx, sy, r)                      # old spot disappears
    pw, ph = patch.size
    dx0, dy0 = max(0, dx-pw//2), max(0, dy-ph//2)
    img.paste(patch, (dx0, dy0))                      # snapped onto destination
    d = ImageDraw.Draw(img)
    d.rectangle([dx0, dy0, dx0+pw, dy0+ph], outline=(60, 220, 120), width=max(3, W//200))
    d.text((dx0, max(0, dy0-14)), "joined", fill=(60, 220, 120))
    return img

print("logic ready")

logic ready


In [ ]:
# 3b. GENERATIVE connect: name the two clicked parts, then EDIT the photo so they are actually joined.
def _fit8(img, maxside=512):
    img = img.convert("RGB")
    w, h = img.size
    s = maxside / max(w, h)
    if s < 1:
        w, h = int(w * s), int(h * s)
    w, h = max(8, (w // 8) * 8), max(8, (h // 8) * 8)
    return img.resize((w, h))

def _name_at(canvas, x, y):
    W, H = canvas.size
    r = min(W, H) // 6
    crop = canvas.crop((max(0, x - r), max(0, y - r), min(W, x + r), min(H, y + r)))
    try:
        return model.query(crop, "Name this object in 1 to 3 words.")["answer"].strip().strip(".")
    except Exception:
        return "part"

def connect_generate(canvas, sx, sy, dx, dy, steps=30, guidance=7.0, image_guidance=1.5):
    src = _name_at(canvas, sx, sy)
    dst = _name_at(canvas, dx, dy)
    instruction = f"connect the {src} into the {dst} so they are properly joined together"
    img = _fit8(canvas)
    out = pipe_edit(
        instruction, image=img,
        num_inference_steps=steps,
        guidance_scale=guidance,          # how strongly to follow the instruction
        image_guidance_scale=image_guidance,  # how much to preserve the original photo
    ).images[0]
    return out, instruction

print("generative connect ready")

In [ ]:
# 4. The manipulable interface (Gradio)

def on_generate(image, task):
    if image is None or not task or not task.strip():
        return None, "Upload an image and type a task first.", [], 0, None, None
    base = image.convert("RGB")
    steps = generate_steps(base, task)
    view, label = render(base, steps, 0)
    # outputs: step_view, label, steps_state, idx_state, base_state, canvas_state
    return view, label, steps, 0, base, base

def nav(delta, steps, idx, canvas):
    if not steps or canvas is None:
        return gr.update(), gr.update(), idx
    idx = max(0, min(len(steps)-1, idx + delta))
    view, label = render(canvas, steps, idx)
    return view, label, idx

def reset_canvas(base):
    if base is None:
        return gr.update(), None, None
    return base, base, None  # step_view, canvas_state, pending_state

def on_click(canvas, mode, pending, evt: gr.SelectData):
    # returns: step_view, deep_img, deep_txt, canvas_state, pending_state
    if canvas is None:
        return gr.update(), gr.update(), "", canvas, None
    x, y = evt.index
    W, H = canvas.size

    if mode == "Connect (press to join)":
        if pending is None:
            img = canvas.convert("RGB").copy()
            d = ImageDraw.Draw(img)
            r = max(6, min(W, H)//45)
            d.ellipse([x-r, y-r, x+r, y+r], fill=(60, 220, 120))
            d.text((x+r+2, y), "now press the destination", fill=(60, 220, 120))
            return img, gr.update(), "Source picked. Press where it should join.", canvas, (x, y)
        sx, sy = pending
        # GENERATE a new image with the two parts actually joined (image-editing model).
        joined, instr = connect_generate(canvas, sx, sy, x, y)
        return joined, gr.update(), "Generated join -> '" + instr + "'", joined, None

    if mode == "Set highlight (correct)":
        img = canvas.convert("RGB").copy()
        d = ImageDraw.Draw(img)
        r = min(W, H) // 8
        d.rectangle([x-r, y-r, x+r, y+r], outline=(60, 160, 255), width=max(3, W//200))
        d.text((max(0, x-r), max(0, y-r-14)), "you set this", fill=(60, 160, 255))
        return img, gr.update(), "Highlight moved by hand (the human grounding signal).", canvas, None

    # Ask about object / deepen
    r = min(W, H) // 6
    crop = canvas.crop((max(0, x-r), max(0, y-r), min(W, x+r), min(H, y+r)))
    cap = model.query(crop, "What is this and what should I do with it here? One short sentence.")["answer"]
    return gr.update(), crop, cap, canvas, None

def on_context(cimg):
    if cimg is None:
        return ""
    note = model.query(cimg.convert("RGB"),
                       "Read any visible text or labels and give the single most useful detail "
                       "for following an instruction. One sentence.")["answer"]
    return "Context added: " + note

with gr.Blocks(title="Instruction interface demo") as demo:
    gr.Markdown("## Image + task -> manipulable manual\nUpload a photo, type a task, **Generate**, then flip / tap / **connect (generates a joined image)** / add context.")
    steps_state = gr.State([])
    idx_state = gr.State(0)
    base_state = gr.State(None)
    canvas_state = gr.State(None)
    pending_state = gr.State(None)

    with gr.Row():
        with gr.Column(scale=2):
            in_img = gr.Image(type="pil", label="Scene photo")
            task = gr.Textbox(label="Task", placeholder="e.g. Put the laptop on charge, I am in Canada")
            go = gr.Button("Generate manual", variant="primary")
            step_label = gr.Markdown("")
            mode = gr.Radio(["Ask about object", "Connect (press to join)", "Set highlight (correct)"],
                            value="Ask about object", label="Click mode")
            step_view = gr.Image(type="pil", label="Working canvas (click / press here)", interactive=False)
            with gr.Row():
                prev = gr.Button("< Prev")
                nxt = gr.Button("Next >")
                rst = gr.Button("Reset canvas")
        with gr.Column(scale=1):
            gr.Markdown("### Tap-to-deepen")
            deep_img = gr.Image(type="pil", label="Zoom", interactive=False)
            deep_txt = gr.Markdown("")
            gr.Markdown("### Context tray\nUpload a close-up (e.g. a label) for more context.")
            ctx_img = gr.Image(type="pil", label="Extra context image")
            ctx_btn = gr.Button("Add context")
            ctx_note = gr.Markdown("")

    go.click(on_generate, [in_img, task],
             [step_view, step_label, steps_state, idx_state, base_state, canvas_state])
    prev.click(lambda s, i, c: nav(-1, s, i, c), [steps_state, idx_state, canvas_state],
               [step_view, step_label, idx_state])
    nxt.click(lambda s, i, c: nav(1, s, i, c), [steps_state, idx_state, canvas_state],
              [step_view, step_label, idx_state])
    rst.click(reset_canvas, [base_state], [step_view, canvas_state, pending_state])
    step_view.select(on_click, [canvas_state, mode, pending_state],
                     [step_view, deep_img, deep_txt, canvas_state, pending_state])
    ctx_btn.click(on_context, [ctx_img], [ctx_note])

demo.launch(share=True, debug=False)

## How to use

1. Run cells top to bottom (GPU + Internet on). Cell 2b downloads the image editor (~first run is slow).
2. Open the public `gradio.live` link from the last cell.
3. Upload a photo, type a task, click **Generate manual**.
4. **Flip** with Prev/Next to walk the grounded steps.
5. **Connect**: set mode to "Connect (press to join)", press the part to move, then press where it
   joins. The model **generates a new image** with the two parts actually connected (takes a few seconds).
6. **Tap-to-deepen**: mode "Ask about object", click anything to crop + explain it.
7. **Correct**: mode "Set highlight", click where the highlight should be.
8. **Context**: upload a close-up, click **Add context**, read the note.
9. **Reset canvas** restores the original photo.

## What this is and is not
- The "join" is now a **generated edit** (InstructPix2Pix): it names the two clicked parts and edits the
  photo so they are joined. This is the real, hard step — and you will see it is often **plausible but not
  physically correct** (parts overlap, lighting off). That failure is exactly the research problem.
- InstructPix2Pix is a small SD1.5-era editor chosen so it co-fits with Moondream on a free T4. For higher
  quality, swap `pipe_edit` in cell 2b for **Qwen-Image-Edit** (needs a bigger GPU), and pass a
  **blueprint/target reference** alongside the instruction — that is the proposed fix for connection correctness.
- Small-model caveat: Moondream's decomposition/detection and the editor will sometimes miss. That is
  expected and is why manual correction and context-upload stay in the loop.